SyntaxError: invalid syntax (1376443489.py, line 1)

In [ ]:

# ---------- ۰. نصب و ایمپورت ----------
%pip install vosk pyaudio requests tqdm --quiet

import pyaudio
import numpy as np
import os
import json
import time
import requests
import zipfile
from tqdm import tqdm
from collections import deque

print("✅ کتابخانه‌ها نصب و ایمپورت شدند.")

Note: you may need to restart the kernel to use updated packages.
✅ کتابخانه‌ها نصب و ایمپورت شدند.


In [3]:
# ---------- ۱. دانلود مدل از گیت‌هاب ----------
MODEL_NAME = "vosk-model-small-en-us-0.15"
MODEL_FILE = f"{MODEL_NAME}.zip"
MODEL_PATH = "model"

# لینک مستقیم دانلود (raw) از مخزن kercre123
MODEL_URL = "https://github.com/kercre123/vosk-models/raw/main/vosk-model-small-en-us-0.15.zip"

def download_model():
    if os.path.exists(MODEL_PATH):
        print(f"✅ مدل از قبل در مسیر '{MODEL_PATH}' موجود است.")
        return True

    print(f"📥 دانلود مدل {MODEL_NAME} از گیت‌هاب...")
    print(f"🔗 {MODEL_URL}")

    try:
        resp = requests.get(MODEL_URL, stream=True, timeout=60)
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0))
        with open(MODEL_FILE, "wb") as f:
            for chunk in tqdm(resp.iter_content(1024), total=total//1024, unit='KB'):
                f.write(chunk)

        print("📂 استخراج فایل...")
        with zipfile.ZipFile(MODEL_FILE, 'r') as zf:
            zf.extractall(".")
        os.remove(MODEL_FILE)
        os.rename(MODEL_NAME, MODEL_PATH)
        print("✅ مدل آماده شد.")
        return True
    except Exception as e:
        print(f"❌ خطا در دانلود: {e}")
        print("🔧 راه‌حل دستی:")
        print("1. این فایل را با مرورگر دانلود کنید:")
        print(f"   {MODEL_URL}")
        print("2. آن را کنار نوت‌بوک اکسترکت کنید.")
        print("3. نام پوشه را به 'model' تغییر دهید.")
        return False

download_model()

📥 دانلود مدل vosk-model-small-en-us-0.15 از گیت‌هاب...
🔗 https://github.com/kercre123/vosk-models/raw/main/vosk-model-small-en-us-0.15.zip


40241KB [00:53, 750.54KB/s]                           


📂 استخراج فایل...
✅ مدل آماده شد.


True

In [ ]:
# ---------- ۲. تنظیمات میکروفون و VAD (Voice Activity Detection) ----------
RATE = 16000
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1

ENERGY_THRESHOLD = 400     # آستانه انرژی (با توجه به نویز محیط قابل تنظیم)
SILENCE_LIMIT = 2.0        # ثانیه سکوت برای اتمام صحبت
PRE_SPEECH_BUFFER_SEC = 0.3

def record_with_buffer(stream):
    pre_len = int(PRE_SPEECH_BUFFER_SEC * RATE / CHUNK) + 1
    pre_buf = deque(maxlen=pre_len)
    frames = []
    started = False
    last_speech_time = time.time()

    print("🤫 ... در حال گوش دادن")
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)
        if not data:
            continue

        audio_np = np.frombuffer(data, dtype=np.int16).astype(np.float64)
        rms = np.sqrt(np.mean(audio_np**2))

        if not started:
            pre_buf.append(data)
            if rms > ENERGY_THRESHOLD:
                print("🗣️ گفتار شناسایی شد – ضبط با بافر...")
                started = True
                frames.extend(pre_buf)
                frames.append(data)
                last_speech_time = time.time()
        else:
            frames.append(data)
            if rms > ENERGY_THRESHOLD:
                last_speech_time = time.time()
            if (time.time() - last_speech_time) > SILENCE_LIMIT:
                print("🔇 سکوت – ضبط متوقف شد.")
                break
    return b''.join(frames)

In [ ]:
# ---------- ۳. بارگذاری مدل و حلقه اصلی بیدارباش ----------
from vosk import Model, KaldiRecognizer

if not os.path.exists(MODEL_PATH):
    print("❌ مدل پیدا نشد. سلول دانلود را دوباره اجرا کنید.")
else:
    print("🧠 بارگذاری مدل Vosk...")
    model = Model(MODEL_PATH)
    rec = KaldiRecognizer(model, RATE)
    print("✅ مدل بارگذاری شد.")

    KEYWORD = "Hello"   # هر کلیدواژه انگلیسی که دوست دارید

    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE,
                    input=True, frames_per_buffer=CHUNK)

    print(f"🚀 سیستم آماده است. بگویید: «{KEYWORD}»")
    print("برای خروج Ctrl+C\n")

    try:
        while True:
            audio_bytes = record_with_buffer(stream)
            if len(audio_bytes) == 0:
                continue

            if rec.AcceptWaveform(audio_bytes):
                text = json.loads(rec.Result()).get("text", "")
                if text:
                    print(f"📝 تشخیص: {text}")
                    if KEYWORD in text.lower():
                        print(f"🔔🔔🔔 بیدارباش! «{KEYWORD}» دریافت شد. 🔔🔔🔔")
    except KeyboardInterrupt:
        print("\n⏹️ توقف با Ctrl+C")
    finally:
        stream.stop_stream()
        stream.close()
        p.terminate()
        print("👋 منابع بسته شدند. خدا نگهدار!")

🧠 بارگذاری مدل Vosk...
✅ مدل بارگذاری شد.
🚀 سیستم آماده است. بگویید: «Hello»
برای خروج Ctrl+C

🤫 ... در حال گوش دادن
🗣️ گفتار شناسایی شد – ضبط با بافر...
🔇 سکوت – ضبط متوقف شد.
📝 تشخیص: what a phenomenon of a phantom
🤫 ... در حال گوش دادن
🗣️ گفتار شناسایی شد – ضبط با بافر...

⏹️ توقف با Ctrl+C
👋 منابع بسته شدند. خدا نگهدار!
